# `frame_builder.py` walkthrough

This notebook is a runnable companion/reference to `frame_builder.py` in this same directory. `frame_builder.py`
visualises reorganised Rietkerk/DP simulation data (see `../Utilities/reorganise_dir.py` and
`../Utilities/reorganise_prelims_dir.py`) as heatmap images/videos, and computes/plots a battery of derived
summary statistics (equilibrium states, FFT power spectra, potential wells, cluster statistics, spatio-temporal
synchrony/correlation, phase-space trajectories, cross-parameter comparisons).

`frame_builder.py`'s own module docstring puts it this way:

> This script visualizes the data in the form of a video. It reads the data from the csv file and creates a
> video of the data. ... The data is stored in a directory structure as follows:
> `{in_dir}/{PREFIX}/L_{g}_a_{a_val}/dP_{dP}/Geq_{Geq}/T_{T}/FRAME_T_{T}_a_{a_val}_R_{R}.csv` where terms in
> `{}` are variables that are provided as input to the script. In these csvs, the columns are as follows:
> `a_c  x  Sp1 ... SpN` where the 2nd column onwards (indexed from 0) are the data columns and represent the
> species concentration (for each species) at each grid point. The columns are of length `{g*g}` (excluding
> the header row) where `g` is the grid size.

**How this notebook relates to the script**: `frame_builder.py`'s own bottom "driver" section (where all the
actual function calls live, several of them commented out as toggleable alternatives) is wrapped in
`if __name__ == "__main__":`, so `import frame_builder` here does **not** re-run any of it — you get the
functions and the always-active "User inputs" globals (`SPB`, `in_dir`, `out_dir`, `g`, `dP`, `Geq`, `R_max`)
for free, and this notebook defines its own copies of the remaining ("secondary") globals
(`a_vals`, `T_vals`, `TS_vals`, `TCorr_vals`, `prefixes`, `a_scaling`, `variable_labels`, ...) itself, one cell
per functional block from the bottom of the script — including the ones that are commented out there. Each
code cell below is a directly-runnable, uncommented version of one such block, preceded by a markdown cell
describing what it does (using `frame_builder.py`'s own `''' # Summary of ... '''`-style comment blocks where
one exists next to the relevant function, and a synthesised description — clearly marked as such — where it
doesn't).

**Important**: some functions defined in `frame_builder.py` read `g`, `dP`, `Geq`, `SPB`, `R_max`, `in_dir` and
`out_dir` as **bare module-level globals** rather than as function parameters (e.g. `frame_visualiser` builds
paths like `f"{in_dir}/{PREFIX}/L_{g}_a_{a}/dP_{dP}/Geq_{Geq}/..."` using the module's own `g`/`dP`/`Geq`, not
anything passed as an argument). Because of this, if you want to change one of *those* seven variables for a
run in this notebook, you must set it via `fb.g = ...` (i.e. mutate the `frame_builder` module's own
namespace) rather than a plain local variable — a plain `g = 128` in this notebook would have **no effect** on
what the imported functions actually do. All the other "secondary" globals below (`a_vals`, `T_vals`, `TS_vals`,
`TCorr_vals`, `prefixes`, `a_scaling`, `variable_labels`, `var_frame_labels`, `move_var_labels`,
`Tavg_win_index`, `Traj_win_index`, `Tavg_win_index_move`, `statType`, `cluststat_filename`, `prelim_fits`,
`compare_list`) are always passed explicitly as function *arguments*, so plain local notebook variables for
those work exactly as you'd expect.

**Working directory**: `frame_builder.py`'s paths (`in_dir`, `out_dir`, etc.) are relative -- run this notebook
from inside `simulations/Data_Processing/`, the same way you'd run the script itself.

## Setup: import `frame_builder` and confirm the primary globals

Importing `frame_builder` runs everything in the file *except* the code below the `if __name__ == "__main__":`
guard -- so this executes the module's own "User inputs" section (setting `SPB`, `in_dir`, `out_dir`, `g`, `dP`,
`Geq`, `R_max` to whatever is currently hardcoded at the top of `frame_builder.py`) and defines every function,
but does not run any video generation/analysis and does not block on the script's own
`input("Press F to pay respects...")` prompt.

In [ ]:
import frame_builder as fb

print("Primary globals imported from frame_builder.py (edit via fb.X = ... to override for this notebook):")
print(f"  fb.SPB    = {fb.SPB}")
print(f"  fb.in_dir = {fb.in_dir}")
print(f"  fb.out_dir= {fb.out_dir}")
print(f"  fb.g      = {fb.g}")
print(f"  fb.dP     = {fb.dP}")
print(f"  fb.Geq    = {fb.Geq}")
print(f"  fb.R_max  = {fb.R_max}  (-1 means auto-detect maxR.txt per T/a)")

# Example overrides (mirrors the commented-out alternative in_dir/out_dir values kept in frame_builder.py itself),
# uncomment and edit as needed -- note these MUST be set on fb.<name>, not as a bare local variable, see note above.
# fb.SPB = 3
# fb.in_dir = f"../Data/Rietkerk/Reorganised_Frames/Stoc/{fb.SPB}Sp/ASCALE_20_100_HsX/"
# fb.out_dir = f"../../Images/{fb.SPB}Sp/ASCALE_20_100_HsX/"
# fb.g = 256; fb.dP = 10000; fb.Geq = 0.19208; fb.R_max = -1

## Secondary globals

These mirror the hardcoded values found in the "driver" section at the bottom of `frame_builder.py` (the
active values there at the time this notebook was written -- edit freely; the many commented-out alternative
values from the script are kept below too, exactly as in the script, for reference). These are passed
explicitly into every function call below, so redefining them here is all that's needed (no `fb.` prefix
required, unlike the primary globals above).

In [ ]:
a_vals = [0.026, 0.042, 0.05, 0.052]
#[1.68, 1.686, 1.688, 1.69, 1.692, 1.694, 1.696, 1.698]
#[0.02, 0.022, 0.023, 0.0235, 0.024, 0.0245, 0.025, 0.0255, 0.026, 0.028, 0.03, 0.034, 0.036, 0.038, 0.0415, 0.042, 0.044, 0.046, 0.05, 0.051, 0.052, 0.053, 0.054, 0.055, 0.06, 0.065, 0.07]

a_scaling = 1  # 0.001 for paper-scale DP-model 'a' values

T_vals = [0, 80000, 84000, 88000, 92000, 96000, 100000, 104000, 108000, 112000, 116000, 120000,
          120226, 131826, 144544, 158489, 173780, 190546]
# A handful of other T_vals lists kept (commented) in frame_builder.py for different dt/spreading-test/gamma-check runs:
#T_vals= [0, 76000, 80000, 82000, 84000, 86000, 88000, 90000, 91201.1]
# TVALS WHEN DT= 0.1: T_vals=[0, 63095.7, 69183, 75857.7, 83176.3, 91201, 100000, 109647, 120226, 131826, 144544, 158489, 173780, 190546]
# TVALS WHEN DT= 0.12: T_vals=[0, 20.88, 30.12, ..., 14454.4]
# TVALS FOR GAMMA CHECK: T_vals= [0, 82000.1, 84000, 86000, 88000, 90000, 92000, 94000, 96000, 98000, 100000, 144544, 158489, 173780, 190546]

prefixes = ["HsX1005-FGPER05-UA125A125-5E2UNI", "HsX1005-FGPER1-UA125A125-5E2UNI", "HsX1005-FGPER2-UA125A125-5E2UNI"]
# Many other prefix lists are kept (commented) in frame_builder.py for other experiment batches, e.g.:
#prefixes = ["HsX2001-FGPER05-UA125A0-5E2UNI", "HsX2001-FGPER1-UA125A0-5E2UNI", "HsX2001-FGPER2-UA125A0-5E2UNI"]
#prefixes = ["CORR-DsB6-UA0A0-1UNI"]
# LOCUSTS: prefixes = ["DsCW10-HXR2010-UA0A0", "DsCW10-HXR2010-UA125A0", "DsC-HXR2010-UA0A0", "DsC-HXR2010-UA125A0", ...]
# GENERIC SMALL MAMMAL: prefixes = ["DsC-HXR03015-UA0A0-1UNI", "DsC-HXR2010-UA0A0-1UNI", ...]

TS_vals = [120226]
TCorr_vals = [4000.04]

print("Note the following set values:")
print(f"Prefixes: {prefixes}")
print(f"TS_vals: {TS_vals}")
print(f"T_vals: {T_vals}")
print(f"a_vals: {a_vals}")
print(f"SPB: {fb.SPB}, g: {fb.g}, dP: {fb.dP}, Geq: {fb.Geq}, maxR: {fb.R_max}")
print(f"in_dir: {fb.in_dir} \nout_dir: {fb.out_dir}")
# NOTE: the script itself pauses here with input("Press F to pay respects...") before proceeding --
# in the notebook, that role is simply played by you choosing when to run the next cell.

## Copy manifest/text files from `in_dir` to `out_dir`

**`recursive_copydir(...)`** ("Summary Description of `recursive_copydir(...)`", from `frame_builder.py`):
> This function recursively copies files from the source directory to the destination directory.
> `src`: source directory. `dst`: destination directory. `include_filetypes`: list of filetypes to be copied.
> `symlinks`: If True, symlinks are copied as symlinks. If False, the files are copied as hardlinks.

In the script this is used to copy the `.txt` manifest files (`a_vals.txt`, `T_vals.txt`, `maxR.txt`, etc. --
but not the `.csv`/image/video data itself) from `in_dir` into `out_dir`, so `out_dir` ends up with its own
copy of the bookkeeping files alongside the generated images/videos/plots.

In [ ]:
fb.recursive_copydir(fb.in_dir, fb.out_dir, include_filetypes=["*.txt"],
                      exclude_filetypes=["*.png", "*.jpg", "*.jpeg", "*.mp4"], symlinks=False)

## Species-count-dependent derived labels

These label lists and averaging-time-windows are re-derived from `fb.SPB` in the script (`if SPB == 3: ...
elif SPB == 2: ... elif SPB == 1: ...`) and are passed into most of the `analyse_*` functions below.
`variable_labels` names the Prelims (time-series) columns, `var_frame_labels` the Frame (per-cell CSV) columns,
`move_var_labels` the movement/gamma Prelims columns.

`frame_builder.py`'s own inline comment on the averaging-window semantics:
> Window for averaging over last `Tavg_win_index` values of T (if negative, then average over last
> `|Tavg_win_index|` values of T). If `Tavg_win_index[0] < 0` and `Tavg_win_index[1] <= 0`, then average over
> last `Tavg_win_index[0] + Tmax` to `Tavg_win_index[1] + Tmax` values of T. If
> `Tavg_win_index[1] >= Tavg_win_index[0] >= 0`, then average over last `Tavg_win_index[0]` to
> `Tavg_win_index[1]` values of T.

In [ ]:
if fb.SPB == 3:
    variable_labels = ["<<P(x; t)>_x>_r", "<<G(x; t)>_x>_r", "<<Pr(x; t)>_x>_r"]
    var_frame_labels = ["P(x; t)", "G(x; t)", "Pr(x; t)"]
    move_var_labels = ["<GAM[G(x; t)]>_x", "<GAM[Pr(x; t)]>_x"]
    Tavg_win_index = [190000, 200000]; Traj_win_index = [100, 3000]; Tavg_win_index_move = [190000, 200000]
elif fb.SPB == 2:
    variable_labels = ["<<P(x; t)>_x>_r", "<<G(x; t)>_x>_r"]
    var_frame_labels = ["P(x; t)", "G(x; t)"]
    move_var_labels = ["<GAM[G(x; t)]>_x"]
    Tavg_win_index = [160000, 200000]; Traj_win_index = [1000, 10000]; Tavg_win_index_move = [80000, 150000]
elif fb.SPB == 1:
    variable_labels = ["<<P(x; t)>_x>_r"]; var_frame_labels = ["P(x; t)"]
    Tavg_win_index = [180000, 200000]; Traj_win_index = [200, 5000]
    move_var_labels = ["<GAM[G(x; t)]>_x"]; Tavg_win_index_move = [120000, 125000]

print(f"variable_labels: {variable_labels}")
print(f"var_frame_labels: {var_frame_labels}")
print(f"move_var_labels: {move_var_labels}")
print(f"Tavg_win_index: {Tavg_win_index}, Traj_win_index: {Traj_win_index}, Tavg_win_index_move: {Tavg_win_index_move}")

## Video generation: all replicates combined into one video per (Prefix, a)

Uses **`frame_visualiser(...)`** to render one heatmap PNG per `(a, T, R)` combination, then
**`home_video(...)`** to stitch every replicate's PNGs for a given `(Prefix, a)` into a single video (so the
video shows all replicates, T-value by T-value).

`home_video`'s own summary ("Summary of `home_video(...)`"):
> This function creates a video of the png data in the directory dir which has the structure:
> `{filedir}/{prefixes}/L_{g}_a_{a_val}/dP_{dP}/Geq_{Geq}/T_{T}/{pngformat}`. The data is stored in the png
> files with the format specified in the `pngformat` parameter. The video is saved in the `out_dir` directory
> with the name specified in the `videoname` parameter. The function reads the data from the png files and
> stores them in a list of images. Optionally, one can provide the relative path (from parendir) to the text
> files delineating `a_vals`, `T_vals` and `maxR` (if these values are not specified/set to 0) by the relative
> paths given by `avals_txtdir`, `Tvals_txtdir` and `maxR_txtdir`. The same applies for `pngdir` (relative path
> to png files for given a, T, dP, Geq etc values).

`frame_visualiser` itself has no separate summary block, but per its own inline comments: if `a_vals`/`T_vals`
are passed as empty lists, it auto-detects and processes every value found in the corresponding
`a_vals.txt`/`T_vals.txt` manifest file rather than a fixed list (`maxR <= 0` does the same for `maxR.txt`).

This is the "FOR VIDEOS OF ALL REPLICATES" block in `frame_builder.py` (commented out there in favour of the
per-replicate variant below; kept here as its own runnable cell).

In [ ]:
for Pre in prefixes:
    fb.frame_visualiser(fb.in_dir, fb.out_dir, Pre, a_vals, T_vals, maxR=-1, plt_gamma=False, delpng=False)
    print(f"Done with making frames for {Pre}")
    for a in a_vals:
        print(f"Making video for {Pre} at a = {a} \n\n")
        fb.home_video(fb.out_dir, fb.out_dir, [Pre], [a], T_vals, maxR=-1,
                      pngformat="CombinedImg/BioConc_a_{a}_T_{T}_n_{R}.png",
                      pngdir="{Pre}/L_{g}_a_{a}/dP_{dP}/Geq_{Geq}/T_{T}/",
                      maxR_txtdir="{Pre}/L_{g}_a_{a}/dP_{dP}/Geq_{Geq}/T_{T}/maxR.txt", videoname="guess",
                      video_relpath="{Pre}/Videos/Conc/{a}/{Tmin}-{Tmax}/")

## Video generation: one video per individual replicate

Same as above, but `frame_visualiser`/`home_video` are called once per replicate index `R` (`maxR=R+1,
minR=R`), so each replicate gets its own separate video rather than all replicates being combined into one.
This is the block that's **active** (not commented out) at the bottom of `frame_builder.py` -- i.e. this is
what actually runs if you execute the script directly with no other changes.

In [ ]:
for Pre in prefixes:
    for R in range(0, 2):
        fb.frame_visualiser(fb.in_dir, fb.out_dir, Pre, a_vals, T_vals, maxR=R + 1, minR=R, plt_gamma=False, delpng=False)
        print(f"Done with making frames for {Pre}")
        for a in a_vals:
            print(f"Making video for {Pre} at a = {a} \n\n")
            fb.home_video(fb.out_dir, fb.out_dir, [Pre], [a], T_vals=T_vals, maxR=R + 1, minR=R,
                          pngformat="CombinedImg/BioConc_a_{a}_T_{T}_n_{R}.png",
                          pngdir="{Pre}/L_{g}_a_{a}/dP_{dP}/Geq_{Geq}/T_{T}/",
                          maxR_txtdir="{Pre}/L_{g}_a_{a}/dP_{dP}/Geq_{Geq}/T_{T}/maxR.txt", videoname="guess",
                          video_relpath="{Pre}/Videos/Conc/{a}/{Tmin}-{Tmax}/")

## Video generation: all replicates, across every `a` value, for each `T`

Same functions again, but the video groups by `T` (one frame per `a` value at that `T`) instead of by `a`
(one frame per `T` value at that `a`) -- useful for seeing how the spatial pattern changes across the control
parameter scan at a fixed point in time, rather than how one `a` value evolves over time.

In [ ]:
for Pre in prefixes:
    fb.frame_visualiser(fb.in_dir, fb.out_dir, Pre, a_vals, T_vals, maxR=-1, plt_gamma=False, delpng=False)
    print(f"Done with making frames for {Pre}")
    fb.home_video(fb.out_dir, fb.out_dir, [Pre], a_vals, T_vals, maxR=-1,
                  pngformat="CombinedImg/BioConc_a_{a}_T_{T}_n_{R}.png",
                  pngdir="{Pre}/L_{g}_a_{a}/dP_{dP}/Geq_{Geq}/T_{T}/",
                  maxR_txtdir="{Pre}/L_{g}_a_{a}/dP_{dP}/Geq_{Geq}/T_{T}/maxR.txt", videoname="guess",
                  video_relpath="{Pre}/Videos/Conc/{amin}-{amax}/{Tmin}-{Tmax}/")

# Minimal-defaults one-off variant kept (commented) in frame_builder.py, using the functions' own built-in
# pngformat/pngdir/video_relpath defaults instead of the explicit format strings above:
#fb.frame_visualiser(fb.in_dir, fb.out_dir, prefixes[0], a_vals, T_vals, maxR=-1, plt_gamma=False, delpng=False)
#fb.home_video(fb.out_dir, fb.out_dir, prefixes, a_vals, T_vals, fb.R_max,
#              pngformat="CombinedImg/BioConc_a_{a}_T_{T}_n_{R}.png", videoname="guess")

## Video generation: "spreading test" videos (with GAMMA overlay)

Same as the per-replicate video block, but `plt_gamma=True` additionally overlays/renders the GAMMA
(interaction-field) data alongside the state-variable concentration, and only a single replicate (`R=0`) is
rendered. In `frame_builder.py` this block is explicitly labelled for prefixes named
`NREF-GAU`/`REF-GAU` (spreading-experiment runs).

In [ ]:
for Pre in prefixes:
    for R in range(0, 1):
        fb.frame_visualiser(fb.in_dir, fb.out_dir, Pre, a_vals, T_vals, maxR=R + 1, minR=R, plt_gamma=True, delpng=False)
    print(f"Done with making frames for {Pre}")
    for a in a_vals:
        print(f"Making video for {Pre} at a = {a} \n\n")
        fb.home_video(fb.out_dir, fb.out_dir, [Pre], [a], T_vals=T_vals, maxR=R + 1, minR=R,
                      pngformat="CombinedImg/BioGammaConc_a_{a}_T_{T}_n_{R}.png",
                      pngdir="{Pre}/L_{g}_a_{a}/dP_{dP}/Geq_{Geq}/T_{T}/",
                      maxR_txtdir="{Pre}/L_{g}_a_{a}/dP_{dP}/Geq_{Geq}/T_{T}/maxR.txt", videoname="guess",
                      video_relpath="{Pre}/Videos/Conc/{a}/{Tmin}-{Tmax}/")

## Frame equilibrium-state analysis (`analyse_FRAME_EQdata`)

Reads each `T`'s `MEAN_STD_Surviving_Runs.txt`/`MEAN_STD_All_Runs.txt` (produced by
`../Utilities/reorganise_dir.py`'s post-processing stage) via the helper `get_FRAME_EQdata`, and plots the
mean +/- std of each species' concentration as a function of the control parameter `a`, averaged over the
`Tavg_window_index` time window -- the standard "equilibrium state diagram" / bifurcation-style plot.

`analyse_FRAME_EQdata` itself has no separate author summary block in `frame_builder.py`, but its data-fetching
helper `get_FRAME_EQdata` does ("Summary Description of `get_FRAME_EQdata(...)` (called by
`analyse_EQdata(...)`)"):
> This function creates a multi-Indexed DataFrame with the equilibrium data for each species. The indices are
> as follows: Prefix, a, T. The columns are determined by reading the header of tab-delineated txt file
> `MEAN_STD_Surviving_runs.txt` which is located in the directory
> `{in_dir}/{PREFIX}/L_{g}_a_{a_val}/dP_{dP}/Geq_{Geq}/T_{T}/`. These column names should be the same for all
> such files in all such directories. The second line of the file contains the values for the columns. All the
> data is stored in the DataFrame and returned.

In [ ]:
fb.analyse_FRAME_EQdata(fb.in_dir, fb.out_dir, prefixes, a_vals, T_vals, Tavg_window_index=Tavg_win_index,
                          filename="MEAN_STD_Surviving_Runs.txt")

## Prelims time-series analysis with power-law fits (`analyse_PRELIMS_TIMESERIESdata`)

Plots each species' Prelims (spatially-averaged) time series (from `MEAN_TSERIES_T_{TS}.csv`), optionally
overlaying a fitted decay/power-law curve (`prelim_fits`) and marker lines at specific `a_c` critical values.

No author summary block exists for this exact function in `frame_builder.py` (there is one for the related,
older `analyse_FRAME_timeseriesData`, quoted for context since the naming/behavior is analogous):
> This function analyses the timeseries data for each species and creates plots for each species for each
> Prefix and a value, with T as the x-axis. NOTE: This function assumes that the data is stored in the
> FILENAME the following format: `R_max AVG[{var}]_SURV ... AVG[{var}]_ALL ... AVG[{var}]_R_0 ...
> AVG[{var}]_R_{R_max}` where `{var}` is one of the species names.

In [ ]:
prelim_fits = {"<<P(x; t)>_x>_r": {"fitFunc": fb.decay_power_lawfit, "exp": 0.451, "A0": None, "xmin": 80000, "xmax": 120000},
               "<<G(x; t)>_x>_r": {"fitFunc": fb.power_lawfit, "exp": None, "A0": None, "xmin": 2000, "xmax": 20000},
               "<<Pr(x; t)>_x>_r": {"fitFunc": fb.power_lawfit, "exp": None, "A0": None, "xmin": 4000, "xmax": 30000}}

fb.analyse_PRELIMS_TIMESERIESdata(fb.in_dir, fb.out_dir, prefixes, a_vals, TS_vals, serType="TSERIES",
                                    meanfilename="MEAN_TSERIES_T_{TS}.csv",
                                    a_c=[1.72, 1.725, 1.7275, 1.73, 1.74, 1.76, 1.78, 1.8], prelim_fit=prelim_fits,
                                    var_labels=variable_labels, a_scaling=a_scaling)

# Simpler variant kept (commented) in frame_builder.py, with no fit overlay / critical-a markers:
#fb.analyse_PRELIMS_TIMESERIESdata(fb.in_dir, fb.out_dir, prefixes, a_vals, TS_vals, serType="TSERIES",
#                                    meanfilename="MEAN_TSERIES_T_{TS}.csv", a_c=None,
#                                    var_labels=variable_labels, a_scaling=a_scaling)

## Prelims equilibrium-state summary + violin plots (`analyse_PRELIMS_EQdata`)

Like `analyse_FRAME_EQdata` above, but for Prelims data: averages each species' Prelims time series over
`Tavg_window_index`, and (with `plt_violin=True`) additionally produces violin plots of the per-replicate
distribution at each `a` value, in addition to the usual mean +/- std vs. `a` line plot. This is also the
function whose output (`DEBUG_*.csv` files under `out_dir/{Pre}/PhaseDiagrams/`) is later consumed by
`multiplot_PRELIMS_CompareEQ` (see near the end of this notebook).

In [ ]:
fb.analyse_PRELIMS_EQdata(fb.in_dir, fb.out_dir, prefixes, a_vals, TS_vals, Tavg_window_index=Tavg_win_index,
                            serType="TSERIES", meanfilename="MEAN_TSERIES_T_{TS}.csv", var_labels=variable_labels,
                            plt_violin=True, a_scaling=a_scaling)

## Phase-space trajectory plots (`analyse_PRELIMS_TRAJECTORYdata`)

Plots trajectories of one species against another over time (e.g. grazer vs. predator concentration), for
each individual replicate as well as a combined plot, with a third species' concentration as the point hue.

("Summary Description of `analyse_PRELIMS_TRAJECTORYdata(...)`"):
> This function analyses the trajectory data for each species and creates plots for each species for each
> Prefix, a value and TS (Time Series) value. These plots are trajectory plots (for each individual replicate)
> of the various species (designated by the `x_label`, `y_label`) over time with the hue and the size of the
> points given by the `hue_label` and `size_label` respectively. ... By plotting the `{var}` columns given by
> the `x_label`, `y_label`, `hue_label` and `size_label`, we can create a trajectory plot for each species for
> each Prefix, a value, TS and R value. ... `reduce_points` (by default `None`) can be set to an integer value
> to reduce the number of points plotted for each trajectory by that factor ... `maxR=-1` will plot all
> encountered R values in data, while `minR=-1` will additionally plot the mean values across all R values.

In [ ]:
for a in a_vals:
    fb.analyse_PRELIMS_TRAJECTORYdata(fb.in_dir, fb.out_dir, prefixes, [a], TS_vals, size_label=["t"], maxR=5, minR=0,
                                        T_window=Traj_win_index, meanfilename="Mean_TSERIES_T_{TS}_dP_{dP}_Geq_{Geq}.csv",
                                        a_scaling=a_scaling, x_label=["<<G(x; t)>_x>_r"], y_label=["<<Pr(x; t)>_x>_r"],
                                        hue_label=["<<P(x; t)>_x>_r"])

## Frame FFT power-spectrum analysis (`analyse_FRAME_FFTPOWERdata`)

Reads each replicate's 2D spatial FFT power spectrum (`FFT_POWERspectra.csv`, produced by
`../Utilities/reorganise_dir.py`'s `gen_FFT_PowerSpectra`) and plots it as a heatmap (frequency on the y-axis,
`a` on the x-axis), averaged over the given `T` window -- this is how you spot a characteristic pattern
wavelength (banding/spot spacing) as a function of the control parameter.

("Summary Description of `analyse_FRAME_FFTPOWERdata(...)`"):
> This function analyses the FFT POWER data for each species and creates plots for each species for each
> Prefix and a value, averaged over provided T values. It creates heatmaps of the power spectra for each
> species for each Prefix and a value, with frequency as the y-axis and a as the x-axis. ... It creates a
> Multi-Index DataFrame with Indices: Prefix, a, T, R and columns: `FREQ[{var}]`, `POWER[{var}]_MEAN_ALL`,
> `POWER[{var}]_MEAN_SURV`, `POWER[{var}]_R_{R_max}`.

In [ ]:
fb.analyse_FRAME_FFTPOWERdata(fb.in_dir, fb.out_dir, prefixes, a_vals, T_vals, filename="FFT_POWERspectra.csv",
                                Tavg_window_index=Tavg_win_index, show_fundamental_freq=True,
                                var_labels=var_frame_labels, a_scaling=a_scaling)

## Frame potential-well / KDE analysis (`analyse_FRAME_POTdata`)

Reads each `T`'s KDE-based `Pot_Well.csv`/`LOCAL_MINIMA.csv` (produced by
`../Utilities/reorganise_dir.py`'s `gen_potential_well_data`) via the helper `get_FRAME_POTdata`, and plots the
estimated potential landscape (and, if `find_minima=True`, marks the local minima) for each state-variable
column as a function of `a` -- the standard way to visualise bistability/alternative-stable-states behavior.

`analyse_FRAME_POTdata` itself has no separate author summary block; its data-fetching helper `get_FRAME_POTdata`
does ("Summary Description of `get_FRAME_POTdata(...)` (called by `analyse_POTdata(...)`)"):
> This function creates a multi-Indexed DataFrame with the potential data for each species. The indices are as
> follows: Prefix, a, T, maxR. The columns are determined by reading the header of comma-delineated csv file
> `Pot_Well.csv` which is located in the directory `{in_dir}/{PREFIX}/L_{g}_a_{a_val}/dP_{dP}/Geq_{Geq}/T_{T}/Pot_Well/`.
> ... Additionally, the local minima is provided in the csv file `LOCAL_MINIMA.csv` located in the same
> directory as `Pot_Well.csv`. If `read_local_minima` is set to True, the local minima data is read and stored
> in another DataFrame and returned. ... Returns: `(data, local_minima)` if `read_local_minima` is set to True,
> else `(data, None)`.

In [ ]:
fb.analyse_FRAME_POTdata(fb.in_dir, fb.out_dir, prefixes, a_vals, T_vals, find_minima=True, filename="Pot_Well.csv",
                           minimafilename="LOCAL_MINIMA.csv", var_labels=["P(x; t)", "G(x; t)", "Pr(x; t)"],
                           a_scaling=a_scaling)

## Frame cluster-size distribution (`analyse_FRAME_CLUSTEREDISTdata`)

Reads each `T`'s `CLUSTERED_FREQUENCIES.csv` (produced by `../Utilities/reorganise_dir.py`'s clustering stage,
`gen_clustered_data` -- here using the `binType="Zero"` threshold-clustering option, see the
`Utilities/README.md` for the `KMeans`/`GMM`/`Zero` choices) and plots the cluster-size frequency distribution
(size on the x-axis, frequency on the y-axis, one line per `a` value), optionally as a CDF.

("Summary description of the `analyse_FRAME_CLUSTEREDISTdata(...)`"):
> Reads frequency distribution of clustered data from csv file given by `filename`, for various prefixes, a
> and T values. Then plots the data as lineplots with size of cluster on the x-axis and frequency on the
> y-axis (a value serving as hue for different lines). This plot is made for each prefix and T value.

In [ ]:
fb.analyse_FRAME_CLUSTEREDISTdata(fb.in_dir, fb.out_dir, prefixes, a_vals, T_vals, filename="CLUSTERED_FREQUENCIES.csv",
                                    clustsubdir="CLUST/{binType}_{nbins}/", binType="Zero", nbins=2,
                                    var_labels=var_frame_labels, plot_binframes=False, CDF=True, a_c=[1.8],
                                    a_scaling=a_scaling)

## Frame-based cluster-statistic time series (`analyse_FRAME_CLUSTERSTATS_timeseriesData`)

For each statistic named in `statType` (mean cluster size, number of clusters, cluster density, occupied-site
fraction, mean squared distance to the grid/cluster center) reads the `MACRO_FRAMESTATS.csv`/`MACRO_CLUSTERSTATS.csv`
files (produced by `../Utilities/reorganise_dir.py`'s `gen_clustered_data`, here using the `binType="GMM"`
option) and plots that statistic as a function of `a`, with an optional decay-power-law fit overlay
(`prelim_fits`, redefined here for the Frame-column names rather than the Prelims-column names used in
Section 7).

("Summary Description of `analyse_FRAME_CLUSTERSTATS_timeseriesData(...)`"):
> This function analyses the cluster statistics timeseries data for each species and creates plots for each
> species for each Prefix and a value, with T as the x-axis. NOTE: This function assumes that the data is
> stored in the FILENAME (located in the directory
> `{in_dir}/{PREFIX}/L_{g}_a_{a_val}/dP_{dP}/Geq_{Geq}/T_{T}/{cluststatsubdir}/`) in the following format:
> `Rmax, a, T, AVG[{statType}{var1}] ... AVG[{statType}{varN}] VAR[{statType}{var1}] ... VAR[{statType}{varN}]
> {statType}{var1}_R_0 ... {statType}{var1}_R_R1 {statType}{var2}_R_0 ... {statType}{varN}_R_RN` where `{var}`
> is one of the species names, and R1, ..., RN are the R values for which the cluster statistics data is
> available (NOTE: `Rmax >= max(R1, ..., RN)`).

In [ ]:
statType = ["MEAN_CLUSTER_SIZE", "NUM_CLUSTERS", "CLUS_DENSITY", "OCCUPIED_SITE_FRAC", "MEAN_SQ_DIST_LCENTER", "MEAN_SQ_DIST_UCOM"]
cluststat_filename = "MACRO_FRAMESTATS.csv"  # or "MACRO_CLUSTERSTATS.csv"
prelim_fits = {"P(x; t)": {"fitFunc": fb.decay_power_lawfit, "exp": 0.451, "A0": None, "xmin": 80000, "xmax": 120000},
               "G(x; t)": {"fitFunc": fb.decay_power_lawfit, "exp": None, "A0": None, "xmin": 2000, "xmax": 20000},
               "Pr(x; t)": {"fitFunc": fb.decay_power_lawfit, "exp": None, "A0": None, "xmin": 5000, "xmax": 100000}}

for stat in statType:
    fb.analyse_FRAME_CLUSTERSTATS_timeseriesData(fb.in_dir, fb.out_dir, prefixes, a_vals, T_vals, statType=stat,
                                                    binType="GMM", nbins=2, filename=cluststat_filename,
                                                    cluststatsubdir="CLUST/{binType}_{nbins}/", a_c="all",
                                                    prelim_fit=prelim_fits, var_labels="deduce", a_scaling=a_scaling)

## Movement/GAMMA Prelims analysis

Same two functions as Sections 7-8 (`analyse_PRELIMS_TIMESERIESdata`, `analyse_PRELIMS_EQdata`), but pointed at
`serType="MOVSERIES"` (the movement/interaction-gamma Prelims files) instead of `"TSERIES"`, and using
`move_var_labels` instead of `variable_labels`.

In [ ]:
fb.analyse_PRELIMS_TIMESERIESdata(fb.in_dir, fb.out_dir, prefixes, a_vals, TS_vals, serType="MOVSERIES",
                                    meanfilename="MEAN_{serType}_T_{TS}.csv", var_labels=move_var_labels, a_scaling=a_scaling)

fb.analyse_PRELIMS_EQdata(fb.in_dir, fb.out_dir, prefixes, a_vals, TS_vals, Tavg_window_index=Tavg_win_index_move,
                            serType="MOVSERIES", meanfilename="MEAN_{serType}_T_{TS}.csv", var_labels=move_var_labels,
                            plt_violin=True, a_scaling=a_scaling)

## Frame potential-well analysis of the GAMMA (interaction) fields

Same function as Section 11 (`analyse_FRAME_POTdata`), but applied to the `GAM[G(x; t)]`/`GAM[Pr(x; t)]`
interaction/movement columns instead of the state-variable columns, saved under a `Gamma_`-prefixed filename.

In [ ]:
fb.analyse_FRAME_POTdata(fb.in_dir, fb.out_dir, prefixes, a_vals, T_vals, find_minima=True, filename="Gamma_Pot_Well.csv",
                           minimafilename="Gamma_LOCAL_MINIMA.csv", var_labels=["GAM[G(x; t)]", "GAM[Pr(x; t)]"])

## Spatio-temporal synchrony/correlation analysis (`analyse_FRAME_CORRdata`)

For each correlation/synchrony statistic in `analType` (NCC, ZNCC, AMI, MI, bivariate Moran's I -- see
`../Utilities/README.md`'s description of `post_imgprocess`/`gen_2DCorr_data`, which is what generates the
underlying `{corrType}_{analType}_TD_*.csv` files this reads), loads and plots the auto- and cross-correlation
data across the `TCorr_vals` timepoints, and optionally builds videos of the resulting plots.

("Summary Description of `analyse_FRAME_CORRdata(...)`"):
> Analyse and visualize 2D auto- and cross-correlation data from simulation outputs. This function processes
> correlation data files for multiple simulation prefixes, computes summary statistics, generates various
> plots (line plots and violin plots) for both auto- and cross-correlation data, saves the results, and
> optionally creates videos from the generated plots. ... Generates and saves line plots and violin plots for
> each variable, a value, and T0 value. Saves processed data as CSV files in the output directory. Optionally
> creates violin plots of mean correlations across replicates for each variable and a value. Returns: None --
> all results are saved to disk; nothing is returned.

In [ ]:
analType = ["NCC", "ZNCC", "AMI", "MI", "BVMoransI"]
for anal in analType:
    fb.analyse_FRAME_CORRdata(fb.in_dir, fb.out_dir, prefixes, a_vals, TCorr_vals, analType=anal,
                                filename="{corrType}_{analType}_TD_*.csv", var_labels="deduce", a_scaling=a_scaling)

## Harmonic-frequency (FFT) analysis of the correlation data (`analyse_FRAME_FFTCORRdata`)

Same idea as the previous section, but reads the `HarmonicPeaks_{corrType}_{analType}_TD_*.csv` files (produced
by `../Utilities/reorganise_dir.py`'s `post_process_df_files` -> `get_1D_HarmonicFreq_Prelimsdata`, i.e. the
1D-FFT peak-finding pass run *over* the lag-correlation curves from the previous section) and plots the
dominant oscillation frequency/period extracted from each correlation curve.

No author summary block exists for this function in `frame_builder.py` -- the description above is inferred
from its parameters (`corrsubdir="2DCorr/FFT/"`, `harm_peaks_index`) and its structural similarity to
`analyse_FRAME_CORRdata`, not an author docstring.

In [ ]:
analType = ["NCC", "ZNCC", "AMI", "MI", "BVMoransI"]
for anal in analType:
    print(f"Analysing {anal} FFT data...")
    fb.analyse_FRAME_FFTCORRdata(fb.in_dir, fb.out_dir, prefixes, a_vals, TCorr_vals, analType=anal, corrsubdir="2DCorr/FFT/",
                                   harm_peaks_index=[0], filename="HarmonicPeaks_{corrType}_{analType}_TD_*.csv",
                                   var_labels="deduce", a_scaling=a_scaling)

## Cross-prefix/parameter comparison plots (`multiplot_PRELIMS_CompareEQ`)

Combines the `DEBUG_*.csv` files written by `analyse_PRELIMS_EQdata` (Section 8) across *multiple* prefixes,
grid sizes, `dP` or `Geq` values into a single comparison plot -- run Section 8 (or an equivalent
`analyse_PRELIMS_EQdata` call for each condition you want to compare) first, so the `DEBUG_*.csv` files this
reads actually exist.

("Summary Description of `multiplot_PRELIMS_CompareEQ(...)`"):
> This function essentially creates combined plots of multiple dataframes generated and saved to csv files by
> the `analyse_PRELIMS_EQdata(...)` function. NOTE: The `analyse_PRELIMS_EQdata(...)` function saves the data
> in the following format: `Savedir = out_dir + f"{Pre}/PhaseDiagrams/"` ... `Filename =
> f"DEBUG_{savefilename}_L_{g}_dP_{dP}_Geq_{Geq}.csv"` ... Given the data saved in this format, the
> `multiplot_PRELIMS_CompareEQ(...)` function creates combined plots of the data for each species, doing so in
> a similar fashion to the `analyse_PRELIMS_EQdata(...)` function, but for multiple `{Pre}`, `{g}`, `{dP}` OR
> `{Geq}` values. ... `compare_list`: A dictionary of the form `{"Prefix": [...], "g": [...], "dP": [...],
> "Geq": [...]}` where the values are lists of the parameters to compare across. ... `meanfilename`: ... If
> provided as `"guess"`, the function will attempt to guess the filename based on comparisons to matching
> files in the directory.

In [ ]:
#compare_list = {"Prefixes": [], "g": [], "dP": [100, 10000], "Geq": []}
compare_list = {"Prefixes": ["HsX2005-UA125A0-5E2UNI", "HsX2005-UA125A125-5E2UNI"], "g": [], "dP": [], "Geq": []}
fb.multiplot_PRELIMS_CompareEQ(fb.out_dir, fb.out_dir, compare_list, prefixes=prefixes, meanfilename="guess",
                                 var_labels=variable_labels)

## Notes

- Several of the analysis stages above (cluster statistics, potential wells, synchrony/correlation) depend on
  post-processing output that `../Utilities/reorganise_dir.py` only generates when its corresponding stage is
  explicitly enabled -- if that post-processing hasn't been run for a given `PREFIX`/`a`/`T` combination, the
  corresponding cell here will find no matching files and report so per-file/per-directory (printed messages,
  no exception) rather than produce a plot; this is expected, not a bug, for a real dataset that hasn't had
  every optional post-processing stage run over it.
- Every function above accepts empty lists for `a_vals`/`T_vals`/`TS_vals`/`TCorr_vals` (and `R_max <= 0`) to
  auto-detect and process every value found on disk, rather than a fixed hand-picked list -- see the "Secondary
  globals" section above.
- This notebook does not call `input(...)` anywhere (unlike the script's own driver section, which pauses once
  before proceeding) -- each cell only runs when you run it, which serves the same purpose.